# Final Information Extraction Evaluation

Clean, reportable evaluation notebook for the six-project benchmark.

**Scope:** 30 frozen PRD/meeting passages across JBIDE, SAK, MNG, MC, HBASE, and SERVER. This notebook loads the frozen human ground truth and the team extraction outputs, evaluates entity extraction and contract-aligned relation extraction, saves detailed CSVs, and displays only the final report tables.

**Reporting rule:** use the final tables produced by the last cell. Relation Evaluation V3 supersedes earlier relation scoring.


In [1]:
# SETUP — run from top in Google Colab
from google.colab import drive
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount("/content/drive")

EXTRACTION_ROOT = Path("/content/drive/MyDrive/GC1/GC1_Entity_Extraction")
KNOWLEDGE_ROOT = Path("/content/drive/MyDrive/01_Knowledge_Construction")
IE_ROOT = KNOWLEDGE_ROOT / "Information_Extraction"
GROUND_TRUTH_DIR = IE_ROOT / "Ground_Truth"
FINAL_ANNOTATION_DIR = GROUND_TRUTH_DIR / "Final_Human_Annotations"
FINAL_RESULTS_DIR = IE_ROOT / "Results" / "FINAL_IE_RESULTS"
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_BENCHMARK_FILE = GROUND_TRUTH_DIR / "FINAL_FROZEN_30_passage_benchmark.csv"
FINAL_ENTITY_FILE = FINAL_ANNOTATION_DIR / "final_entity_ground_truth.csv"
FINAL_RELATION_FILE = FINAL_ANNOTATION_DIR / "final_relation_ground_truth.csv"

required = [FROZEN_BENCHMARK_FILE, FINAL_ENTITY_FILE, FINAL_RELATION_FILE]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required frozen evaluation files:\n" + "\n".join(missing))

final_benchmark = pd.read_csv(FROZEN_BENCHMARK_FILE)
final_entity_ground_truth = pd.read_csv(FINAL_ENTITY_FILE)
final_relation_ground_truth = pd.read_csv(FINAL_RELATION_FILE)

EVAL_PROJECTS = {
    "JBIDE": "02_JBIDE/JBIDE-I01",
    "SAK": "03_SAK/SAK-I01",
    "MNG": "05_MNG/MNG-I01",
    "MC": "10_MC/MC-I01",
    "HBASE": "18_HBASE/HBASE-I01",
    "SERVER": "20_SERVER/SERVER-I01",
}
SOURCE_FILES = {"PRD": "prd_extracted.jsonl", "MEETING": "meetings_extracted.jsonl"}

ENTITY_TYPES = ["PROJECT","ISSUE","PERSON","COMPONENT","FEATURE","SYSTEM","REQUIREMENT","TEAM","MILESTONE","MEETING"]
RELATION_TYPES = ["HAS_ISSUE","ASSIGNED_TO","REPORTED_BY","CREATED_BY","HAS_COMPONENT","RELATED_TO","DEPENDS_ON","BLOCKED_BY","BLOCKS","MENTIONS","REFERENCES_ISSUE","HAS_REQUIREMENT","HAS_MILESTONE","HAS_STAKEHOLDER","HAS_MEETING","PARTICIPATED_IN","REFERENCES_REQUIREMENT"]

# Silent integrity checks
assert len(final_benchmark) == 30, f"Expected 30 benchmark passages, found {len(final_benchmark)}"
assert set(final_benchmark['sample_id']) == set(f"FINAL-{i:03d}" for i in range(1,31))
assert set(final_entity_ground_truth['sample_id']).issubset(set(final_benchmark['sample_id']))
assert set(final_relation_ground_truth['sample_id']).issubset(set(final_benchmark['sample_id']))
assert set(final_entity_ground_truth['entity_type']).issubset(set(ENTITY_TYPES))
assert set(final_relation_ground_truth['relation_type']).issubset(set(RELATION_TYPES))


Mounted at /content/drive


In [2]:
# HELPERS
def norm(text):
    if pd.isna(text): return ""
    text = str(text).strip().lower()
    text = text.replace("’", "'").replace("“", '"').replace("”", '"').replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text)

def source_id_from_filename(filename):
    return str(filename).removesuffix(".txt")

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: yield json.loads(line)

def metric_counts(h, m):
    tp=len(h&m); fp=len(m-h); fn=len(h-m)
    p=tp/(tp+fp) if tp+fp else np.nan
    r=tp/(tp+fn) if tp+fn else np.nan
    f=2*p*r/(p+r) if pd.notna(p) and pd.notna(r) and (p+r) else (0.0 if tp+fp+fn else np.nan)
    return {'TP':tp,'FP':fp,'FN':fn,'precision':p,'recall':r,'f1':f}

def subset_set_by_samples(values, sample_ids):
    ids=set(sample_ids); return {x for x in values if x[0] in ids}

def subgroup_metrics(group_col, human_set, model_set, label):
    rows=[]
    for value,grp in final_benchmark.groupby(group_col):
        ids=grp['sample_id'].tolist(); h=subset_set_by_samples(human_set,ids); m=subset_set_by_samples(model_set,ids)
        rows.append({group_col:value,'task':label,'support':len(h),**metric_counts(h,m)})
    return pd.DataFrame(rows)

def bootstrap_metric(human_set, model_set, n_boot=2000, seed=42):
    rng=np.random.default_rng(seed); ids=final_benchmark['sample_id'].tolist(); vals=[]
    h_by={sid:[x[1:] for x in human_set if x[0]==sid] for sid in ids}
    m_by={sid:[x[1:] for x in model_set if x[0]==sid] for sid in ids}
    for _ in range(n_boot):
        tp=fp=fn=0
        for sid in rng.choice(ids,size=len(ids),replace=True):
            h=set(h_by[sid]); m=set(m_by[sid]); tp+=len(h&m); fp+=len(m-h); fn+=len(h-m)
        p=tp/(tp+fp) if tp+fp else np.nan; r=tp/(tp+fn) if tp+fn else np.nan
        f=2*p*r/(p+r) if pd.notna(p) and pd.notna(r) and p+r else 0.0
        vals.append((p,r,f))
    a=np.asarray(vals,float)
    return {'precision_low':np.nanpercentile(a[:,0],2.5),'precision_high':np.nanpercentile(a[:,0],97.5),
            'recall_low':np.nanpercentile(a[:,1],2.5),'recall_high':np.nanpercentile(a[:,1],97.5),
            'f1_low':np.nanpercentile(a[:,2],2.5),'f1_high':np.nanpercentile(a[:,2],97.5)}

PROJECT_ALIASES={
    'JBIDE':{'jbide','jboss tools'}, 'SAK':{'sak','sakai'},
    'MNG':{'mng','maven','maven core','apache maven'}, 'MC':{'mc','minecraft: java edition'},
    'HBASE':{'hbase'}, 'SERVER':{'server','core server'}
}


In [3]:
# ENTITY EVALUATION — strict document + passage matching
model_rows=[]
for project,relative_path in EVAL_PROJECTS.items():
    folder=EXTRACTION_ROOT/relative_path
    for source_type,filename in SOURCE_FILES.items():
        for record in read_jsonl(folder/filename):
            source_id=record.get('source',{}).get('source_id')
            for e in record.get('entities',[]):
                text=e.get('name') or e.get('text') or e.get('entity_text'); typ=e.get('type') or e.get('entity_type')
                if text and typ:
                    model_rows.append({'project':project,'document_type':source_type,'source_id':source_id,'entity_text':text,'entity_type':typ,'normalized_text':norm(text),'confidence':e.get('confidence'),'extraction_method':e.get('extraction_method')})
model_entities=pd.DataFrame(model_rows)

preds=[]
for _,s in final_benchmark.iterrows():
    sid=source_id_from_filename(s['document'])
    cand=model_entities[(model_entities.project==s.project)&(model_entities.document_type==s.document_type)&(model_entities.source_id==sid)]
    passage=norm(s['passage'])
    for _,p in cand.iterrows():
        if p.normalized_text and p.normalized_text in passage:
            preds.append({'sample_id':s.sample_id,'project':s.project,'document_type':s.document_type,'document':s.document,'source_id':sid,**p[['entity_text','entity_type','normalized_text','confidence','extraction_method']].to_dict()})
predicted_entities=pd.DataFrame(preds).drop_duplicates(subset=['sample_id','normalized_text','entity_type'])

def canonical_entity_key(project,text,entity_type):
    return norm(project) if entity_type=='PROJECT' else norm(text)

human_entities=final_entity_ground_truth.copy(); model_entities_eval=predicted_entities.copy()
human_entities['eval_text']=human_entities.apply(lambda r:canonical_entity_key(r.project,r.entity_text,r.entity_type),axis=1)
model_entities_eval['eval_text']=model_entities_eval.apply(lambda r:canonical_entity_key(r.project,r.entity_text,r.entity_type),axis=1)
human_typed=set(zip(human_entities.sample_id,human_entities.eval_text,human_entities.entity_type))
model_typed=set(zip(model_entities_eval.sample_id,model_entities_eval.eval_text,model_entities_eval.entity_type))
human_mentions=set(zip(human_entities.sample_id,human_entities.eval_text)); model_mentions=set(zip(model_entities_eval.sample_id,model_entities_eval.eval_text))
typed_results=metric_counts(human_typed,model_typed); mention_results=metric_counts(human_mentions,model_mentions)

entity_type_rows=[]
for t in ENTITY_TYPES:
    h={x for x in human_typed if x[2]==t}; m={x for x in model_typed if x[2]==t}
    entity_type_rows.append({'entity_type':t,'support':len(h),**metric_counts(h,m)})
entity_type_metrics=pd.DataFrame(entity_type_rows)

method_rows=[]
for method,grp in model_entities_eval.groupby(model_entities_eval['extraction_method'].fillna('unknown')):
    pred=set(zip(grp.sample_id,grp.eval_text,grp.entity_type)); tp=len(pred&human_typed); fp=len(pred-human_typed)
    method_rows.append({'extraction_method':method,'predictions':len(pred),'typed_TP':tp,'typed_FP':fp,'prediction_precision':tp/(tp+fp) if tp+fp else np.nan})
method_metrics=pd.DataFrame(method_rows)

semantic_types={'FEATURE','SYSTEM','COMPONENT','TEAM'}
family_rows=[]
for family,types in [('Semantic technical',semantic_types),('Explicit/other',set(ENTITY_TYPES)-semantic_types)]:
    h={x for x in human_typed if x[2] in types}; m={x for x in model_typed if x[2] in types}
    family_rows.append({'family':family,'support':len(h),**metric_counts(h,m)})
entity_family_metrics=pd.DataFrame(family_rows)


In [4]:
# RELATION EVALUATION V3 — production-contract aligned
model_relation_rows=[]
for project,relative_path in EVAL_PROJECTS.items():
    folder=EXTRACTION_ROOT/relative_path
    for source_type,filename in SOURCE_FILES.items():
        for record in read_jsonl(folder/filename):
            source_id=record.get('source',{}).get('source_id')
            lookup={e.get('local_id'):{'name':e.get('name'),'type':e.get('type')} for e in record.get('entities',[]) if e.get('local_id')}
            for rel in record.get('relationships',[]):
                sub=lookup.get(rel.get('source_entity')); obj=lookup.get(rel.get('target_entity')); typ=rel.get('type')
                if sub and obj and typ:
                    model_relation_rows.append({'project':project,'document_type':source_type,'source_id':source_id,'subject_text':sub['name'],'subject_type':sub['type'],'relation_type':typ,'object_text':obj['name'],'object_type':obj['type']})
model_relations=pd.DataFrame(model_relation_rows)

PRD_CONTRACT={'HAS_REQUIREMENT':('PROJECT','REQUIREMENT'),'HAS_MILESTONE':('PROJECT','MILESTONE'),'HAS_STAKEHOLDER':('PROJECT','TEAM'),'REFERENCES_ISSUE':('PROJECT','ISSUE'),'MENTIONS':('PROJECT',None)}
MEETING_CONTRACT={'HAS_MEETING':('PROJECT','MEETING'),'PARTICIPATED_IN':('PERSON','MEETING'),'REFERENCES_ISSUE':('MEETING','ISSUE'),'REFERENCES_REQUIREMENT':('MEETING','REQUIREMENT'),'MENTIONS':('MEETING',None)}

def is_grouped_requirement(text): return bool(re.fullmatch(r'req-\d{3}-\d{3}',norm(text)))
def canon_rel(text,typ,project): return norm(project) if typ=='PROJECT' else norm(text)

human_rel=final_relation_ground_truth.copy(); human_rel['excluded']=False
mask=(human_rel.document_type=='PRD')&(human_rel.relation_type=='HAS_REQUIREMENT')
human_rel.loc[mask,'subject_text']=human_rel.loc[mask,'project']; human_rel.loc[mask,'subject_type']='PROJECT'
range_mask=mask & human_rel.object_text.map(is_grouped_requirement); human_rel.loc[range_mask,'excluded']=True
human_rel_scored=human_rel[~human_rel.excluded].copy()
human_rel_scored['subject_eval']=human_rel_scored.apply(lambda r:canon_rel(r.subject_text,r.subject_type,r.project),axis=1)
human_rel_scored['object_eval']=human_rel_scored.apply(lambda r:canon_rel(r.object_text,r.object_type,r.project),axis=1)

def endpoint_present(text,typ,sample,relation_type):
    text_n=norm(text); passage=norm(sample['passage']); dt=sample['document_type']
    if dt=='PRD' and typ=='PROJECT': return True
    if dt=='MEETING' and typ=='MEETING': return (text_n in passage) if relation_type=='HAS_MEETING' else True
    if dt=='MEETING' and typ=='PROJECT': return any(norm(a) in passage for a in PROJECT_ALIASES.get(sample['project'],set()))
    return bool(text_n) and text_n in passage

rel_preds=[]
for _,s in final_benchmark.iterrows():
    source_id=source_id_from_filename(s.document)
    cand=model_relations[(model_relations.project==s.project)&(model_relations.document_type==s.document_type)&(model_relations.source_id==source_id)]
    contract=PRD_CONTRACT if s.document_type=='PRD' else MEETING_CONTRACT
    for _,r in cand.iterrows():
        if r.relation_type not in contract: continue
        es,eo=contract[r.relation_type]
        if es and r.subject_type!=es: continue
        if eo and r.object_type!=eo: continue
        if endpoint_present(r.subject_text,r.subject_type,s,r.relation_type) and endpoint_present(r.object_text,r.object_type,s,r.relation_type):
            rel_preds.append({'sample_id':s.sample_id,'project':s.project,'document_type':s.document_type,'subject_text':r.subject_text,'subject_type':r.subject_type,'relation_type':r.relation_type,'object_text':r.object_text,'object_type':r.object_type,'subject_eval':canon_rel(r.subject_text,r.subject_type,s.project),'object_eval':canon_rel(r.object_text,r.object_type,s.project)})
predicted_relations=pd.DataFrame(rel_preds).drop_duplicates(subset=['sample_id','subject_eval','relation_type','object_eval'])

human_relation_set=set(zip(human_rel_scored.sample_id,human_rel_scored.subject_eval,human_rel_scored.relation_type,human_rel_scored.object_eval))
model_relation_set=set(zip(predicted_relations.sample_id,predicted_relations.subject_eval,predicted_relations.relation_type,predicted_relations.object_eval))
relation_results=metric_counts(human_relation_set,model_relation_set)

relation_type_rows=[]
for t in RELATION_TYPES:
    h={x for x in human_relation_set if x[2]==t}; m={x for x in model_relation_set if x[2]==t}
    row={'relation_type':t,'support':len(h),**metric_counts(h,m)}; row['status']='EVALUATED' if h else 'N/A'
    relation_type_rows.append(row)
relation_type_metrics=pd.DataFrame(relation_type_rows)


In [5]:
# FINAL ANALYSES + SAVE (silent)
entity_project=pd.concat([subgroup_metrics('project',human_mentions,model_mentions,'Mention detection'),subgroup_metrics('project',human_typed,model_typed,'Typed entity')],ignore_index=True)
relation_project=subgroup_metrics('project',human_relation_set,model_relation_set,'Relation V3')
project_metrics=pd.concat([entity_project,relation_project],ignore_index=True)
entity_source=pd.concat([subgroup_metrics('document_type',human_mentions,model_mentions,'Mention detection'),subgroup_metrics('document_type',human_typed,model_typed,'Typed entity')],ignore_index=True)
relation_source=subgroup_metrics('document_type',human_relation_set,model_relation_set,'Relation V3')
source_metrics=pd.concat([entity_source,relation_source],ignore_index=True)

summary_rows=[]
for name,h,m,res in [('Entity mention detection',human_mentions,model_mentions,mention_results),('Typed entity extraction',human_typed,model_typed,typed_results),('Relation extraction V3',human_relation_set,model_relation_set,relation_results)]:
    summary_rows.append({'evaluation':name,**res,**bootstrap_metric(h,m)})
overall_metrics=pd.DataFrame(summary_rows)

# Save all reportable tables
overall_metrics.to_csv(FINAL_RESULTS_DIR/'overall_metrics.csv',index=False)
entity_type_metrics.to_csv(FINAL_RESULTS_DIR/'entity_metrics_by_type.csv',index=False)
relation_type_metrics.to_csv(FINAL_RESULTS_DIR/'relation_metrics_by_type.csv',index=False)
project_metrics.to_csv(FINAL_RESULTS_DIR/'metrics_by_project.csv',index=False)
source_metrics.to_csv(FINAL_RESULTS_DIR/'metrics_by_source.csv',index=False)
method_metrics.to_csv(FINAL_RESULTS_DIR/'entity_prediction_quality_by_method.csv',index=False)
entity_family_metrics.to_csv(FINAL_RESULTS_DIR/'entity_metrics_by_family.csv',index=False)
predicted_entities.to_csv(FINAL_RESULTS_DIR/'passage_entity_predictions.csv',index=False)
predicted_relations.to_csv(FINAL_RESULTS_DIR/'passage_relation_predictions_v3.csv',index=False)


## Final reportable results

The following tables contain the evaluation results.

- **Entity mention detection** measures whether the mention was found, ignoring type.
- **Typed entity extraction** requires both the mention and entity type to match.
- **Relation extraction V3** uses the production relation contract and supersedes all earlier relation scores.
- Relation types with no human support are omitted from the reportable per-type table rather than shown as zero.
- Results apply to the six-project, 30-passage evaluation benchmark; they are not an estimate over all 30 source projects.


In [6]:
# DISPLAY ONLY FINAL REPORTABLE RESULTS
pd.set_option('display.max_rows',100)

final_overall=overall_metrics[['evaluation','TP','FP','FN','precision','recall','f1','f1_low','f1_high']].copy()
final_overall.columns=['Evaluation','TP','FP','FN','Precision','Recall','F1','F1 95% CI low','F1 95% CI high']
print('FINAL OVERALL INFORMATION EXTRACTION RESULTS')
display(final_overall.round(3))

print('\nENTITY PERFORMANCE BY TYPE')
display(entity_type_metrics[['entity_type','support','precision','recall','f1']].sort_values('f1',ascending=False).round(3))

print('\nRELATION PERFORMANCE BY TYPE — EVALUATED TYPES ONLY')
display(relation_type_metrics[relation_type_metrics.status=='EVALUATED'][['relation_type','support','precision','recall','f1']].sort_values('f1',ascending=False).round(3))

print('\nPRD VS MEETING')
display(source_metrics[['document_type','task','support','precision','recall','f1']].sort_values(['task','document_type']).round(3))

print('\nBY PROJECT')
display(project_metrics[['project','task','support','precision','recall','f1']].sort_values(['task','project']).round(3))

print('\nENTITY PREDICTION QUALITY BY EXTRACTION METHOD')
display(method_metrics[['extraction_method','predictions','typed_TP','typed_FP','prediction_precision']].sort_values('prediction_precision',ascending=False).round(3))

print('\nENTITY PERFORMANCE BY TYPE FAMILY')
display(entity_family_metrics[['family','support','precision','recall','f1']].round(3))

print(f'\nBenchmark: {len(final_benchmark)} passages | Human entities: {len(final_entity_ground_truth)} | Human relations (original): {len(final_relation_ground_truth)} | Relations scored in V3: {len(human_relation_set)}')
print(f'Results saved to: {FINAL_RESULTS_DIR}')


FINAL OVERALL INFORMATION EXTRACTION RESULTS


,Evaluation,TP,FP,FN,Precision,Recall,F1,F1 95% CI low,F1 95% CI high
0,Entity mention detection,156,119,127,0.567,0.551,0.559,0.481,0.629
1,Typed entity extraction,141,146,142,0.491,0.498,0.495,0.408,0.570
2,Relation extraction V3,129,126,71,0.506,0.645,0.567,0.472,0.642



ENTITY PERFORMANCE BY TYPE


,entity_type,support,precision,recall,f1
2,PERSON,84,0.911,0.857,0.883
1,ISSUE,40,1.000,0.700,0.824
9,MEETING,12,1.000,0.500,0.667
0,PROJECT,10,0.417,1.000,0.588
6,REQUIREMENT,17,0.500,0.353,0.414
8,MILESTONE,8,0.500,0.250,0.333
5,SYSTEM,30,0.132,0.300,0.184
7,TEAM,17,0.136,0.176,0.154
3,COMPONENT,42,0.182,0.095,0.125
4,FEATURE,23,0.045,0.043,0.044



RELATION PERFORMANCE BY TYPE — EVALUATED TYPES ONLY


...


ENTITY PREDICTION QUALITY BY EXTRACTION METHOD


,extraction_method,predictions,typed_TP,typed_FP,prediction_precision
1,regex,32,32,0,1.000
2,structured,115,92,23,0.800
0,gliner,140,17,123,0.121



ENTITY PERFORMANCE BY TYPE FAMILY


,family,support,precision,recall,f1
0,Semantic technical,112,0.127,0.152,0.138
1,Explicit/other,171,0.810,0.725,0.765



Benchmark: 30 passages | Human entities: 300 | Human relations (original): 219 | Relations scored in V3: 200
Results saved to: /content/drive/MyDrive/01_Knowledge_Construction/Information_Extraction/Results/FINAL_IE_RESULTS
